In [1]:
# ============================================================
# BASIC GAN FOR IMAGE GENERATION (FASHION-MNIST)
# ============================================================

import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, Sequential
from tensorflow.keras.datasets import fashion_mnist
from tensorflow.keras.optimizers import Adam

# ============================================================
# USER CONFIGURATION (INPUT PARAMETERS)
# ============================================================

dataset_choice = "fashion"      # fixed as Fashion-MNIST
epochs = 50                     # recommended 30–100
batch_size = 128                # 64 or 128
noise_dim = 100                 # 50 or 100
learning_rate = 0.0002
save_interval = 5               # save every 5 epochs

# ============================================================
# LOAD FASHION-MNIST DATASET
# ============================================================

(x_train, _), (_, _) = fashion_mnist.load_data()

# Normalize images to [-1, 1]
x_train = x_train.astype("float32")
x_train = (x_train - 127.5) / 127.5
x_train = np.expand_dims(x_train, axis=-1)

img_shape = (28, 28, 1)

# ============================================================
# GENERATOR MODEL
# ============================================================

def build_generator():
    model = Sequential()

    model.add(layers.Dense(256, input_dim=noise_dim))
    model.add(layers.LeakyReLU(0.2))

    model.add(layers.Dense(512))
    model.add(layers.LeakyReLU(0.2))

    model.add(layers.Dense(1024))
    model.add(layers.LeakyReLU(0.2))

    model.add(layers.Dense(28 * 28 * 1, activation="tanh"))
    model.add(layers.Reshape(img_shape))

    return model

# ============================================================
# DISCRIMINATOR MODEL
# ============================================================

def build_discriminator():
    model = Sequential()

    model.add(layers.Flatten(input_shape=img_shape))
    model.add(layers.Dense(512))
    model.add(layers.LeakyReLU(0.2))

    model.add(layers.Dense(256))
    model.add(layers.LeakyReLU(0.2))

    model.add(layers.Dense(1, activation="sigmoid"))

    return model

# ============================================================
# BUILD AND COMPILE MODELS
# ============================================================

optimizer = Adam(learning_rate, 0.5)

discriminator = build_discriminator()
discriminator.compile(
    loss="binary_crossentropy",
    optimizer=optimizer,
    metrics=["accuracy"]
)

generator = build_generator()

# ============================================================
# COMBINED GAN MODEL
# ============================================================

z = layers.Input(shape=(noise_dim,))
img = generator(z)

discriminator.trainable = False

validity = discriminator(img)

gan = tf.keras.Model(z, validity)
gan.compile(loss="binary_crossentropy", optimizer=optimizer)

# ============================================================
# IMAGE SAVING FUNCTION (5x5 GRID)
# ============================================================

def save_images(epoch):
    rows, cols = 5, 5
    noise = np.random.normal(0, 1, (25, noise_dim))
    gen_imgs = generator.predict(noise)

    gen_imgs = (gen_imgs + 1) / 2  # scale to [0,1]

    fig, axs = plt.subplots(rows, cols, figsize=(5,5))
    cnt = 0

    for i in range(rows):
        for j in range(cols):
            axs[i, j].imshow(gen_imgs[cnt, :, :, 0], cmap="gray")
            axs[i, j].axis("off")
            cnt += 1

    os.makedirs("generated_samples", exist_ok=True)
    plt.savefig(f"generated_samples/epoch_{epoch:02d}.png")
    plt.close()

# ============================================================
# TRAINING LOOP
# ============================================================

real = np.ones((batch_size, 1))
fake = np.zeros((batch_size, 1))

print("\n========== TRAINING STARTED ==========\n")

for epoch in range(1, epochs + 1):

    # ---------------------
    # Train Discriminator
    # ---------------------

    idx = np.random.randint(0, x_train.shape[0], batch_size)
    real_imgs = x_train[idx]

    noise = np.random.normal(0, 1, (batch_size, noise_dim))
    fake_imgs = generator.predict(noise)

    d_loss_real = discriminator.train_on_batch(real_imgs, real)
    d_loss_fake = discriminator.train_on_batch(fake_imgs, fake)

    d_loss = 0.5 * np.add(d_loss_real, d_loss_fake)

    # ---------------------
    # Train Generator
    # ---------------------

    noise = np.random.normal(0, 1, (batch_size, noise_dim))
    g_loss = gan.train_on_batch(noise, real)

    print(
        f"Epoch {epoch}/{epochs} | "
        f"D_loss: {d_loss[0]:.3f} | "
        f"D_acc: {d_loss[1]*100:.2f}% | "
        f"G_loss: {g_loss:.3f}"
    )

    if epoch % save_interval == 0:
        save_images(epoch)

# ============================================================
# FINAL IMAGE GENERATION (100 IMAGES)
# ============================================================

os.makedirs("final_generated_images", exist_ok=True)

noise = np.random.normal(0, 1, (100, noise_dim))
final_images = generator.predict(noise)
final_images = (final_images + 1) / 2

for i in range(100):
    plt.imshow(final_images[i, :, :, 0], cmap="gray")
    plt.axis("off")
    plt.savefig(f"final_generated_images/img_{i+1}.png")
    plt.close()

print("\n✅ Training Complete")
print("📁 generated_samples/ → intermediate outputs")
print("📁 final_generated_images/ → 100 synthetic images saved")


29515/29515 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
26421880/26421880 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
5148/5148 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
4422102/4422102 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


/usr/local/lib/python3.12/dist-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)



========== TRAINING STARTED ==========

4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step 


/usr/local/lib/python3.12/dist-packages/keras/src/backend/tensorflow/trainer.py:83: UserWarning: The model does not have any trainable weights.
  warnings.warn("The model does not have any trainable weights.")


Epoch 1/50 | D_loss: 0.662 | D_acc: 63.48% | G_loss: 0.715
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step
Epoch 2/50 | D_loss: 0.696 | D_acc: 56.05% | G_loss: 0.592
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step
Epoch 3/50 | D_loss: 0.767 | D_acc: 47.41% | G_loss: 0.495
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step
Epoch 4/50 | D_loss: 0.854 | D_acc: 43.53% | G_loss: 0.419
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step
Epoch 5/50 | D_loss: 0.954 | D_acc: 40.24% | G_loss: 0.360
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 103ms/step
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
Epoch 6/50 | D_loss: 1.054 | D_acc: 38.46% | G_loss: 0.313
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step
Epoch 7/50 | D_loss: 1.160 | D_acc: 37.15% | G_loss: 0.275
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step
Epoch 8/50 | D_loss: 1.260 | D_acc: 36.48% | G_loss: 0.245
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step
Epoch 9/50 | D_loss: 1.368 | D_acc: 35.16% | G_loss: 0.221
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step
Epoch 10/50 | D_loss: 1.463 | D_acc: 35.04% | G_loss: 0.201
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 